# Problem 4b: Sequence-Based Variational Autoencoder for Speech Emotion Recognition
**By: Jason Clark**

---

## Overview

This notebook implements a **Sequence VAE** using LSTM encoder/decoder architecture to learn continuous latent representations of emotional speech. Unlike the CNN-based VAE in `02_vae_emotion_space.ipynb` which operates on mel-spectrograms as images, this approach:

1. **Extracts MFCC sequences** - 40 coefficients over time, preserving temporal structure
2. **Uses Bidirectional LSTM encoder** - Captures both forward and backward temporal dependencies
3. **Learns a 64-dimensional latent space** - Where similar emotions cluster together
4. **Reconstructs sequences with LSTM decoder** - Validates learned representations

### Why Sequence VAE?
- **Temporal modeling**: LSTMs naturally handle variable-length sequences and temporal dependencies
- **Better for speech**: Speech is inherently sequential; capturing dynamics is crucial for emotion
- **Richer representations**: Bidirectional encoding captures context from both past and future

### Loss Function
```
VAE_Loss = Reconstruction_Loss (MSE) + β * KL_Divergence
```
We use β-VAE formulation with β=0.5 to balance reconstruction quality and latent space regularization.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import glob
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Loading

Load RAVDESS emotional speech audio files from Google Drive. The dataset contains 1440 audio files with 8 emotions:
- neutral, calm, happy, sad, angry, fearful, disgust, surprised

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Data directory in Google Drive
DATA_DIR = Path("/content/drive/MyDrive/Speech-Emotion-Recognition-/data")

# RAVDESS emotion mapping (position 3 in filename)
EMOTION_MAP = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}

# Collect all audio files
audio_files = []
for actor_dir in DATA_DIR.glob("Actor_*"):
    for wav_file in actor_dir.glob("*.wav"):
        parts = wav_file.stem.split("-")
        if len(parts) >= 3:
            emotion_code = parts[2]
            if emotion_code in EMOTION_MAP:
                audio_files.append({
                    'file_path': str(wav_file),
                    'emotion_code': emotion_code,
                    'emotion_label': EMOTION_MAP[emotion_code],
                    'actor': wav_file.parent.name
                })

df = pd.DataFrame(audio_files)
print(f"Total audio files: {len(df)}")
print(f"\nEmotion distribution:")
print(df['emotion_label'].value_counts())

## 3. MFCC Sequence Extraction

Extract MFCC (Mel-Frequency Cepstral Coefficients) sequences from audio files:
- **40 MFCC coefficients** per time frame
- **Variable length sequences** padded/truncated to fixed T=150 timesteps
- MFCCs are better for capturing speech characteristics than raw spectrograms

In [ ]:
# Audio processing parameters
SAMPLE_RATE = 22050
N_MFCC = 40          # Number of MFCC coefficients
N_FFT = 2048         # FFT window size
HOP_LENGTH = 512     # Hop length for STFT
MAX_TIMESTEPS = 150  # Fixed sequence length (pad/truncate)

def extract_mfcc_sequence(file_path):
    """
    Extract MFCC sequence from audio file.
    
    Returns:
        mfcc: numpy array of shape (T, N_MFCC) where T is timesteps
        actual_length: original length before padding/truncation
    """
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        
        # Extract MFCCs
        mfcc = librosa.feature.mfcc(
            y=y, sr=sr, 
            n_mfcc=N_MFCC, 
            n_fft=N_FFT, 
            hop_length=HOP_LENGTH
        )
        
        # Transpose to (T, N_MFCC)
        mfcc = mfcc.T
        actual_length = mfcc.shape[0]
        
        # Pad or truncate to fixed length
        if mfcc.shape[0] < MAX_TIMESTEPS:
            pad_width = MAX_TIMESTEPS - mfcc.shape[0]
            mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:MAX_TIMESTEPS, :]
            actual_length = MAX_TIMESTEPS
        
        return mfcc, actual_length
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, 0

# Extract features for all files
print("Extracting MFCC sequences...")
features = []
lengths = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    mfcc, length = extract_mfcc_sequence(row['file_path'])
    if mfcc is not None:
        features.append(mfcc)
        lengths.append(length)
        labels.append(row['emotion_label'])

# Convert to arrays
X = np.array(features, dtype=np.float32)
lengths = np.array(lengths)
y = np.array(labels)

print(f"\nFeatures shape: {X.shape}")
print(f"  - {X.shape[0]} samples")
print(f"  - {X.shape[1]} timesteps")
print(f"  - {X.shape[2]} MFCC coefficients")
print(f"\nSequence length statistics:")
print(f"  - Min: {lengths.min()}")
print(f"  - Max: {lengths.max()}")
print(f"  - Mean: {lengths.mean():.1f}")

### 3.1 Feature Normalization

Normalize MFCC features to have zero mean and unit variance across the dataset.

In [ ]:
# Compute global mean and std for normalization
X_flat = X.reshape(-1, N_MFCC)
mfcc_mean = X_flat.mean(axis=0)
mfcc_std = X_flat.std(axis=0) + 1e-8  # Add epsilon to avoid division by zero

# Normalize
X_normalized = (X - mfcc_mean) / mfcc_std

print(f"MFCC mean range: [{mfcc_mean.min():.2f}, {mfcc_mean.max():.2f}]")
print(f"MFCC std range: [{mfcc_std.min():.2f}, {mfcc_std.max():.2f}]")
print(f"Normalized data range: [{X_normalized.min():.2f}, {X_normalized.max():.2f}]")

## 4. PyTorch Dataset and DataLoader

In [ ]:
class MFCCSequenceDataset(Dataset):
    """
    PyTorch Dataset for MFCC sequences.
    """
    def __init__(self, sequences, labels, lengths):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = labels
        self.lengths = torch.LongTensor(lengths)
        
        # Encode labels
        self.label_encoder = LabelEncoder()
        self.encoded_labels = torch.LongTensor(
            self.label_encoder.fit_transform(labels)
        )
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return (
            self.sequences[idx], 
            self.encoded_labels[idx],
            self.lengths[idx]
        )
    
    def get_label_names(self):
        return self.label_encoder.classes_

In [ ]:
# Train/test split (stratified)
X_train, X_test, y_train, y_test, len_train, len_test = train_test_split(
    X_normalized, y, lengths,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Create datasets
train_dataset = MFCCSequenceDataset(X_train, y_train, len_train)
test_dataset = MFCCSequenceDataset(X_test, y_test, len_test)

# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

# Full dataset loader for latent space extraction
full_dataset = MFCCSequenceDataset(X_normalized, y, lengths)
full_loader = DataLoader(
    full_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

print(f"\nLabel classes: {train_dataset.get_label_names()}")
print(f"Number of classes: {len(train_dataset.get_label_names())}")

## 5. Sequence VAE Architecture

The Sequence VAE consists of:

1. **Bidirectional LSTM Encoder**: Processes MFCC sequences and outputs hidden states
2. **Latent Space**: Final hidden states → μ and log(σ²) → reparameterization → z
3. **LSTM Decoder**: Reconstructs MFCC sequence from latent vector z

### Architecture Diagram
```
Input: (batch, T, 40)  →  BiLSTM Encoder  →  [h_forward, h_backward]
                                                    ↓
                                            Linear → μ (64-dim)
                                            Linear → log(σ²) (64-dim)
                                                    ↓
                                           Reparameterization
                                                    ↓
                                              z (64-dim)
                                                    ↓
                          Repeat T times → LSTM Decoder → Linear
                                                    ↓
                                        Output: (batch, T, 40)
```

In [ ]:
class SequenceVAE(nn.Module):
    """
    Sequence-based Variational Autoencoder with LSTM encoder and decoder.
    
    Args:
        input_dim: Number of input features per timestep (N_MFCC)
        hidden_dim: LSTM hidden dimension
        latent_dim: Latent space dimension
        num_layers: Number of LSTM layers
        seq_length: Fixed sequence length
    """
    def __init__(self, input_dim=40, hidden_dim=128, latent_dim=64, 
                 num_layers=2, seq_length=150):
        super(SequenceVAE, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        self.seq_length = seq_length
        
        # ============== ENCODER ==============
        # Bidirectional LSTM encoder
        self.encoder_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        
        # Encoder output: concatenate forward and backward final hidden states
        # Output dimension: hidden_dim * 2 (bidirectional) * num_layers
        encoder_output_dim = hidden_dim * 2
        
        # Latent space projection layers
        self.fc_mu = nn.Linear(encoder_output_dim, latent_dim)
        self.fc_logvar = nn.Linear(encoder_output_dim, latent_dim)
        
        # ============== DECODER ==============
        # Project latent to decoder initial hidden state
        self.fc_decode = nn.Linear(latent_dim, hidden_dim * num_layers)
        
        # LSTM decoder (unidirectional)
        self.decoder_lstm = nn.LSTM(
            input_size=latent_dim,  # Input is repeated latent vector
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        
        # Output projection layer
        self.fc_out = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        """
        Encode input sequence to latent distribution parameters.
        
        Args:
            x: Input tensor of shape (batch, seq_length, input_dim)
            
        Returns:
            mu: Mean of latent distribution (batch, latent_dim)
            logvar: Log variance of latent distribution (batch, latent_dim)
        """
        # LSTM encoding
        _, (h_n, _) = self.encoder_lstm(x)
        # h_n shape: (num_layers * 2, batch, hidden_dim) for bidirectional
        
        # Take the final layer's hidden states (forward and backward)
        # Forward: h_n[-2], Backward: h_n[-1]
        h_forward = h_n[-2]  # (batch, hidden_dim)
        h_backward = h_n[-1]  # (batch, hidden_dim)
        
        # Concatenate forward and backward hidden states
        h_combined = torch.cat([h_forward, h_backward], dim=1)  # (batch, hidden_dim * 2)
        
        # Project to latent distribution parameters
        mu = self.fc_mu(h_combined)
        logvar = self.fc_logvar(h_combined)
        
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + sigma * epsilon
        
        Args:
            mu: Mean of latent distribution
            logvar: Log variance of latent distribution
            
        Returns:
            z: Sampled latent vector
        """
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu  # Use mean during evaluation
    
    def decode(self, z):
        """
        Decode latent vector to reconstructed sequence.
        
        Args:
            z: Latent vector of shape (batch, latent_dim)
            
        Returns:
            Reconstructed sequence of shape (batch, seq_length, input_dim)
        """
        batch_size = z.size(0)
        
        # Initialize hidden state from latent
        h_decode = self.fc_decode(z)  # (batch, hidden_dim * num_layers)
        h_decode = h_decode.view(batch_size, self.num_layers, self.hidden_dim)
        h_decode = h_decode.permute(1, 0, 2).contiguous()  # (num_layers, batch, hidden_dim)
        
        # Initialize cell state with zeros
        c_decode = torch.zeros_like(h_decode)
        
        # Repeat latent vector for each timestep
        z_repeated = z.unsqueeze(1).repeat(1, self.seq_length, 1)  # (batch, seq_length, latent_dim)
        
        # LSTM decoding
        output, _ = self.decoder_lstm(z_repeated, (h_decode, c_decode))
        
        # Project to output dimension
        reconstructed = self.fc_out(output)  # (batch, seq_length, input_dim)
        
        return reconstructed
    
    def forward(self, x):
        """
        Forward pass through VAE.
        
        Args:
            x: Input sequence of shape (batch, seq_length, input_dim)
            
        Returns:
            reconstructed: Reconstructed sequence
            mu: Latent mean
            logvar: Latent log variance
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        
        return reconstructed, mu, logvar
    
    def get_latent(self, x):
        """
        Get latent representation for input.
        
        Args:
            x: Input sequence
            
        Returns:
            z: Latent vector (using mean, no sampling)
        """
        mu, logvar = self.encode(x)
        return mu  # Return mean for deterministic latent representation

In [ ]:
# Model hyperparameters
INPUT_DIM = N_MFCC      # 40
HIDDEN_DIM = 128        # LSTM hidden size
LATENT_DIM = 64         # Latent space dimension
NUM_LAYERS = 2          # Number of LSTM layers
SEQ_LENGTH = MAX_TIMESTEPS  # 150

# Instantiate model
model = SequenceVAE(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_layers=NUM_LAYERS,
    seq_length=SEQ_LENGTH
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Loss Function and Training

### β-VAE Loss
We use β-VAE formulation with β=0.5 to balance reconstruction and regularization:

```
Loss = MSE_reconstruction + β * KL_divergence
```

Where:
- **Reconstruction Loss**: MSE between input and output sequences
- **KL Divergence**: Regularizes latent space to be close to N(0, I)

In [ ]:
def vae_loss(reconstructed, original, mu, logvar, beta=0.5):
    """
    Compute β-VAE loss.
    
    Args:
        reconstructed: Reconstructed sequence (batch, seq_len, input_dim)
        original: Original sequence (batch, seq_len, input_dim)
        mu: Latent mean (batch, latent_dim)
        logvar: Latent log variance (batch, latent_dim)
        beta: KL divergence weight
        
    Returns:
        total_loss: Combined loss
        recon_loss: Reconstruction loss only
        kl_loss: KL divergence only
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(reconstructed, original, reduction='mean')
    
    # KL divergence: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss

In [ ]:
# Training hyperparameters
EPOCHS = 100
LEARNING_RATE = 1e-3
BETA = 0.5  # β-VAE weight
PATIENCE = 15  # Early stopping patience

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

def train_epoch(model, train_loader, optimizer, beta):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    for batch_x, _, _ in train_loader:
        batch_x = batch_x.to(device)
        
        # Forward pass
        reconstructed, mu, logvar = model(batch_x)
        
        # Compute loss
        loss, recon_loss, kl_loss = vae_loss(
            reconstructed, batch_x, mu, logvar, beta
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    
    n_batches = len(train_loader)
    return total_loss/n_batches, total_recon/n_batches, total_kl/n_batches

def validate_epoch(model, val_loader, beta):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        for batch_x, _, _ in val_loader:
            batch_x = batch_x.to(device)
            
            reconstructed, mu, logvar = model(batch_x)
            loss, recon_loss, kl_loss = vae_loss(
                reconstructed, batch_x, mu, logvar, beta
            )
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
    
    n_batches = len(val_loader)
    return total_loss/n_batches, total_recon/n_batches, total_kl/n_batches

In [ ]:
# Training history
history = {
    'train_loss': [], 'train_recon': [], 'train_kl': [],
    'val_loss': [], 'val_recon': [], 'val_kl': []
}

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print(f"Training Sequence VAE for {EPOCHS} epochs...")
print(f"β = {BETA}")
print("-" * 70)

for epoch in range(EPOCHS):
    # Train
    train_loss, train_recon, train_kl = train_epoch(
        model, train_loader, optimizer, BETA
    )
    
    # Validate
    val_loss, val_recon, val_kl = validate_epoch(
        model, test_loader, BETA
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_recon'].append(train_recon)
    history['train_kl'].append(train_kl)
    history['val_loss'].append(val_loss)
    history['val_recon'].append(val_recon)
    history['val_kl'].append(val_kl)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} (Recon: {train_recon:.4f}, KL: {train_kl:.4f}) | "
              f"Val Loss: {val_loss:.4f}")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model with validation loss: {best_val_loss:.4f}")

## 7. Training Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_title('Total Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction loss
axes[1].plot(history['train_recon'], label='Train', linewidth=2)
axes[1].plot(history['val_recon'], label='Validation', linewidth=2)
axes[1].set_title('Reconstruction Loss (MSE)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# KL divergence
axes[2].plot(history['train_kl'], label='Train', linewidth=2)
axes[2].plot(history['val_kl'], label='Validation', linewidth=2)
axes[2].set_title('KL Divergence', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Loss', fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_vae_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Latent Space Extraction

In [ ]:
def extract_latent_representations(model, dataloader, device):
    """
    Extract latent representations for all samples.
    
    Returns:
        latent_vectors: numpy array of shape (N, latent_dim)
        labels: numpy array of shape (N,)
    """
    model.eval()
    latent_vectors = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y, _ in dataloader:
            batch_x = batch_x.to(device)
            
            # Get latent representation (mean)
            z = model.get_latent(batch_x)
            
            latent_vectors.append(z.cpu().numpy())
            all_labels.append(batch_y.numpy())
    
    latent_vectors = np.concatenate(latent_vectors, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    return latent_vectors, all_labels

# Extract latent representations for all data
print("Extracting latent representations...")
Z_all, y_encoded_all = extract_latent_representations(model, full_loader, device)

# Get string labels
label_names = full_dataset.get_label_names()
y_labels_all = [label_names[i] for i in y_encoded_all]

print(f"Latent vectors shape: {Z_all.shape}")
print(f"Latent space statistics:")
print(f"  - Mean: {Z_all.mean():.4f}")
print(f"  - Std: {Z_all.std():.4f}")
print(f"  - Min: {Z_all.min():.4f}")
print(f"  - Max: {Z_all.max():.4f}")

## 9. Latent Space Visualization (t-SNE)

In [ ]:
# Apply t-SNE to reduce latent space to 2D for visualization
print("Computing t-SNE projection...")
tsne = TSNE(
    n_components=2, 
    perplexity=30, 
    random_state=SEED,
    n_iter=1000,
    learning_rate='auto',
    init='pca'
)

Z_tsne = tsne.fit_transform(Z_all)
print(f"t-SNE shape: {Z_tsne.shape}")

In [ ]:
# Create visualization
plt.figure(figsize=(14, 10))

# Define colors for emotions
emotion_colors = {
    'angry': '#e41a1c',
    'calm': '#4daf4a',
    'disgust': '#984ea3',
    'fearful': '#ff7f00',
    'happy': '#ffff33',
    'neutral': '#a65628',
    'sad': '#377eb8',
    'surprised': '#f781bf'
}

# Plot each emotion
for emotion in label_names:
    mask = np.array(y_labels_all) == emotion
    plt.scatter(
        Z_tsne[mask, 0], 
        Z_tsne[mask, 1],
        c=emotion_colors.get(emotion, '#999999'),
        label=emotion.capitalize(),
        alpha=0.7,
        s=60,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('Sequence VAE Latent Space (t-SNE)', fontsize=16, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(loc='upper right', fontsize=10, framealpha=0.9)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_vae_latent_space_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Emotion Classification from Latent Space

Evaluate how well the learned latent representations capture emotion-relevant information by training classifiers on the latent space.

In [ ]:
# Extract latent representations for train and test sets separately
Z_train, y_train_encoded = extract_latent_representations(model, train_loader, device)
Z_test, y_test_encoded = extract_latent_representations(model, test_loader, device)

# Get string labels
y_train_labels = [label_names[i] for i in y_train_encoded]
y_test_labels = [label_names[i] for i in y_test_encoded]

print(f"Training latent vectors: {Z_train.shape}")
print(f"Test latent vectors: {Z_test.shape}")

### 10.1 Logistic Regression Classifier

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression classifier...")
lr_classifier = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0
)

lr_classifier.fit(Z_train, y_train_labels)

# Predictions
y_pred_lr = lr_classifier.predict(Z_test)

# Accuracy
acc_lr = accuracy_score(y_test_labels, y_pred_lr)
print(f"\nLogistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.2f}%)")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_lr))

### 10.2 MLP Classifier

In [ ]:
# Train MLP Classifier
print("Training MLP classifier...")
mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=500,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate_init=0.001,
    alpha=0.001  # L2 regularization
)

mlp_classifier.fit(Z_train, y_train_labels)

# Predictions
y_pred_mlp = mlp_classifier.predict(Z_test)

# Accuracy
acc_mlp = accuracy_score(y_test_labels, y_pred_mlp)
print(f"\nMLP Classifier Accuracy: {acc_mlp:.4f} ({acc_mlp*100:.2f}%)")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_mlp))

## 11. Confusion Matrix Visualization

In [ ]:
# Use the better classifier for confusion matrix
best_pred = y_pred_mlp if acc_mlp > acc_lr else y_pred_lr
best_acc = max(acc_mlp, acc_lr)
best_name = "MLP" if acc_mlp > acc_lr else "Logistic Regression"

# Compute confusion matrix
cm = confusion_matrix(y_test_labels, best_pred, labels=label_names)

# Normalize
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=[e.capitalize() for e in label_names],
    yticklabels=[e.capitalize() for e in label_names],
    ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix ({best_name})\nAccuracy: {best_acc:.2%}', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)

# Normalized
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.2f', 
    cmap='Blues',
    xticklabels=[e.capitalize() for e in label_names],
    yticklabels=[e.capitalize() for e in label_names],
    ax=axes[1]
)
axes[1].set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)

plt.tight_layout()
plt.savefig('sequence_vae_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Reconstruction Quality Visualization

In [ ]:
# Get some test samples
model.eval()
with torch.no_grad():
    test_batch_x, test_batch_y, _ = next(iter(test_loader))
    test_batch_x = test_batch_x.to(device)
    reconstructed, _, _ = model(test_batch_x)

# Convert to numpy
original = test_batch_x.cpu().numpy()
reconstructed = reconstructed.cpu().numpy()

# Plot original vs reconstructed for 4 samples
fig, axes = plt.subplots(4, 2, figsize=(14, 12))

for i in range(4):
    # Original
    axes[i, 0].imshow(original[i].T, aspect='auto', origin='lower', cmap='viridis')
    axes[i, 0].set_title(f'Original - {label_names[test_batch_y[i]].capitalize()}', 
                         fontsize=12, fontweight='bold')
    axes[i, 0].set_ylabel('MFCC Coefficient', fontsize=10)
    if i == 3:
        axes[i, 0].set_xlabel('Time Frame', fontsize=10)
    
    # Reconstructed
    axes[i, 1].imshow(reconstructed[i].T, aspect='auto', origin='lower', cmap='viridis')
    axes[i, 1].set_title(f'Reconstructed', fontsize=12, fontweight='bold')
    if i == 3:
        axes[i, 1].set_xlabel('Time Frame', fontsize=10)

plt.suptitle('Original vs Reconstructed MFCC Sequences', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('sequence_vae_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Results Summary

In [ ]:
print("=" * 70)
print("SEQUENCE VAE RESULTS SUMMARY")
print("=" * 70)
print(f"\n📊 Dataset:")
print(f"   - Total samples: {len(full_dataset)}")
print(f"   - Training samples: {len(train_dataset)}")
print(f"   - Test samples: {len(test_dataset)}")
print(f"   - Number of emotions: {len(label_names)}")
print(f"   - Emotions: {', '.join(label_names)}")

print(f"\n🏗️ Model Architecture:")
print(f"   - Input: MFCC sequences ({MAX_TIMESTEPS} timesteps × {N_MFCC} coefficients)")
print(f"   - Encoder: Bidirectional LSTM ({NUM_LAYERS} layers, {HIDDEN_DIM} hidden units)")
print(f"   - Latent dimension: {LATENT_DIM}")
print(f"   - Decoder: LSTM ({NUM_LAYERS} layers, {HIDDEN_DIM} hidden units)")
print(f"   - Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\n📈 Training:")
print(f"   - Epochs trained: {len(history['train_loss'])}")
print(f"   - β (KL weight): {BETA}")
print(f"   - Best validation loss: {best_val_loss:.4f}")
print(f"   - Final reconstruction loss: {history['val_recon'][-1]:.4f}")
print(f"   - Final KL divergence: {history['val_kl'][-1]:.4f}")

print(f"\n🎯 Classification Performance (on latent space):")
print(f"   - Logistic Regression accuracy: {acc_lr:.4f} ({acc_lr*100:.2f}%)")
print(f"   - MLP Classifier accuracy: {acc_mlp:.4f} ({acc_mlp*100:.2f}%)")
print(f"   - Best classifier: {best_name}")

print(f"\n📁 Saved files:")
print(f"   - sequence_vae_training_curves.png")
print(f"   - sequence_vae_latent_space_tsne.png")
print(f"   - sequence_vae_confusion_matrix.png")
print(f"   - sequence_vae_reconstruction.png")
print("=" * 70)

## 14. Conclusions & Key Findings

### Summary

This notebook implemented a **Sequence-based Variational Autoencoder** for speech emotion recognition using:
- **MFCC sequences** as input (40 coefficients × 150 timesteps)
- **Bidirectional LSTM encoder** to capture temporal dependencies
- **64-dimensional latent space** learned with β-VAE formulation
- **LSTM decoder** for sequence reconstruction

### Key Takeaways

1. **Sequence modeling matters**: Unlike the CNN-based approach that treats spectrograms as images, the LSTM architecture naturally handles the temporal nature of speech signals.

2. **Bidirectional encoding**: Capturing both forward and backward context helps create richer emotion representations.

3. **Latent space quality**: The t-SNE visualization shows how well different emotions cluster in the learned latent space.

4. **Classification performance**: The accuracy on the latent space demonstrates the emotion-discriminative power of the learned representations.

### Potential Improvements

- **Attention mechanisms**: Add attention to focus on emotionally salient time frames
- **Contrastive learning**: Incorporate contrastive loss to improve emotion separation
- **Data augmentation**: Apply time stretching, pitch shifting, noise injection
- **Multi-modal features**: Combine MFCCs with other features (spectral, prosodic)

---

*Once you get this back, let me know any feedback so I can improve for next time!*

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_dim': INPUT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'latent_dim': LATENT_DIM,
        'num_layers': NUM_LAYERS,
        'seq_length': SEQ_LENGTH
    },
    'mfcc_mean': mfcc_mean,
    'mfcc_std': mfcc_std,
    'label_names': label_names,
    'history': history,
    'best_val_loss': best_val_loss
}, 'sequence_vae_model.pt')

print("Model saved to sequence_vae_model.pt")